# 01 · Data Cleaning
Load the raw Titanic data, inspect its quality, fix problems, and save a clean copy to `data/processed/`.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/titanic.csv')
print('Shape:', df.shape)
df.head()

Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 1. Structure and data types

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


## 2. Missing values

In [3]:
missing = df.isna().sum()
pd.DataFrame({'missing': missing, 'percent': (missing/len(df)*100).round(1)}).query('missing > 0')

,missing,percent
Age,177,19.9
Cabin,687,77.1
Embarked,2,0.2


**Plan**
- `Age` (~20% missing): fill with the **median age of the passenger's title group** (Mr, Mrs, Miss, Master...), since age differs a lot by title.
- `Embarked` (2 missing): fill with the **mode**.
- `Cabin` (~77% missing): too sparse to impute, so create a `HasCabin` flag and drop the column.

In [4]:
# Extract title from Name
df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.', expand=False).str.strip()
rare = df['Title'].value_counts()[lambda s: s < 10].index
df['Title'] = df['Title'].replace(rare, 'Rare').replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})
df['Title'].value_counts()

Title
Mr        517
Miss      182
Mrs       125
Master     40
Rare       27
Name: count, dtype: int64

In [5]:
df['Age'] = df['Age'].fillna(df.groupby('Title')['Age'].transform('median'))
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['HasCabin'] = df['Cabin'].notna().astype(int)
df = df.drop(columns=['Cabin'])
print('Remaining missing values:', df.isna().sum().sum())

Remaining missing values: 0


## 3. Duplicates

In [6]:
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate PassengerIds:', df['PassengerId'].duplicated().sum())

Duplicate rows: 0
Duplicate PassengerIds: 0


## 4. Outliers in `Fare` (IQR method)

In [7]:
q1, q3 = df['Fare'].quantile([.25,.75])
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
n_out = (df['Fare'] > upper).sum()
print(f'Fare IQR bounds: {lower:.2f} to {upper:.2f}; {n_out} outliers above upper bound')
# Fares are real (first-class tickets), so we keep them but add a log-transformed column for modelling/plots
df['LogFare'] = np.log1p(df['Fare'])

Fare IQR bounds: -26.72 to 65.63; 116 outliers above upper bound


## 5. Feature engineering

In [8]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df['AgeGroup'] = pd.cut(df['Age'], bins=[0,12,18,35,60,100], labels=['Child','Teen','Young Adult','Adult','Senior'])
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,Title,HasCabin,LogFare,FamilySize,IsAlone,AgeGroup
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,Mr,0,2.110213,2,0,Young Adult
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,Mrs,1,4.280593,2,0,Adult
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,Miss,0,2.188856,1,1,Young Adult
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S,Mrs,1,3.990834,2,0,Young Adult
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S,Mr,0,2.202765,1,1,Young Adult


## 6. Save cleaned data

In [9]:
df.to_csv('../data/processed/titanic_clean.csv', index=False)
print('Saved', df.shape)

Saved (891, 17)
